# Test polytopes' volume computation

## Librairies

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Go to project root (adjust depth if needed)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data")
sys.path.append(ROOT)

print("Added to path:", ROOT)

import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset

from src.models.networks import FashionMLP_Large
from src.quantization.quantize import quantize_model
from src.shortcuts.shortcut_weights import *
from src.optim.build_polytopes import *
from src.optim.prune_constraints import *
from src.optim.compute_volumes import estimate_polytope_width


Added to path: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


In [2]:
torch.manual_seed(42)

## Large MLP (FashionMNIST)

In [3]:
# -------------------- #
# Load models and data #
# -------------------- #

# Device
device = torch.device("gpu" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

# Model
model_name = "fashion_mlp_best.pth"
model_path = os.path.join(ROOT, "checkpoints", model_name)

model = FashionMLP_Large()
model.load_state_dict(torch.load(model_path))#['model_state'])
model.to(device).eval()

# qModel
qmodel = quantize_model(model, bits=4)
qmodel.to(device).eval()

# Dataset (and sample)
train_dataset = torch.load(
    DATA_PATH + "/fashionMNIST_correct_mlp.pt",
    weights_only=False
)

x_0, c = train_dataset[13]
print(x_0.min().item(), x_0.max().item()) # check bounds
x_0 = x_0.flatten().unsqueeze(0) # shape (1, input_dim)
x_0 = x_0.to(device) # important
print("Sample x_0 shape:", x_0.shape)
print("Models and dataset have been loaded.")

-1.0 1.0
Sample x_0 shape: torch.Size([1, 784])
Models and dataset have been loaded.


In [4]:
%%time

# --------------------------- #
# Compute volume of polytopes #
# --------------------------- #

print("*** Computing volumes of polytopes... ***\n\n")

nb_directions = 5

bounds = [(-1., 1.)]

A_correct, b_correct, A_both, b_both = build_two_class_polytopes(model, qmodel, x_0, c)

results = estimate_polytope_width(A_correct, b_correct, bounds, 
                                    n_directions=nb_directions,
                                    verbose=True)
results

*** Computing volumes of polytopes... ***


Direction 0: w_correct=37.0142
Direction 1: w_correct=37.6358
Direction 2: w_correct=37.5336
Direction 3: w_correct=37.0908
Direction 4: w_correct=37.3183
CPU times: user 24.1 s, sys: 817 ms, total: 25 s
Wall time: 25.1 s


{'mean_width_correct': 37.318548737934776,
 'std_width_correct': 0.24142268287190685,
 'n_directions_used': 5}

In [5]:
results = estimate_polytope_width(A_both, b_both, bounds, 
                                    n_directions=nb_directions,
                                    verbose=True)

Direction 0: w_correct=36.9410
Direction 1: w_correct=37.7014
Direction 2: w_correct=37.6227
Direction 3: w_correct=37.4682
Direction 4: w_correct=36.8758


In [6]:
torch.argmax(model(x_0)), torch.argmax(qmodel(x_0))

(tensor(7, device='mps:0'), tensor(7, device='mps:0'))